# Level 4 · AI Model Scaling & Deployment — lab notebook
**BIM41 Advanced Predictive Analytics · Dr. Sridevi Tandley · Bharathidasan Institute of Management**

Syllabus hands-on: **deploy a Prophet model as an API endpoint** and **forecast IoT sensor data at scale with PySpark**,
then monitor drift and automate retraining.

| Part | What you build | Deck act |
|---|---|---|
| 1 | Train, package and version a Prophet model (with lineage) | Act 2 |
| 2 | A FastAPI forecast endpoint, tested like production code | Act 3 |
| 3 | The Dockerfile and CI workflow that ship it | Act 3 |
| 4 | PySpark: batch ingestion, features at scale, 50 device forecasts in parallel | Act 1 |
| 5 | Streaming ingestion with Spark Structured Streaming | Act 1 |
| 6 | Drift monitoring: PSI, KS test, Evidently | Act 4 |
| 7 | A retraining pipeline with a champion–challenger gate (Prefect) | Act 4 |

All data is **synthetic** with fixed seeds. Spark needs Java; Google Colab has it. Each Spark and Evidently cell checks first and explains what to install if something is missing.

> **Library note (November 2026):** Evidently 0.7 fails to import with pandas 3. If you see `ImportError: cannot import name 'infer_dtype_from_object'`, run `%pip install "pandas<3"` and restart the kernel. The PSI and KS checks in Part 6 work on any pandas.

In [1]:
# Run this once if a library is missing. "pandas<3" matters: as of November 2026 Evidently 0.7 does not import with pandas 3.
# %pip install -q "pandas<3" prophet fastapi httpx pytest pyspark pyarrow evidently prefect
import warnings, os, sys, json, hashlib, subprocess, logging
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
plt.rcParams["figure.figsize"] = (11, 4)
print("python", sys.version.split()[0], "| pandas", pd.__version__)

python 3.12.3 | pandas 2.3.3


## Part 1 · Train, package and version the model
We reuse the Level 3 retail sales series (same generator, same seed) and train **Prophet** with three regressors.
A deployable model is more than weights: it ships with **metadata** (version, training window, metrics) and a **data fingerprint** for lineage.

In [2]:
def make_retail(seed=7):
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2016-01-01", "2025-12-01", freq="MS")
    t = np.arange(len(idx))
    festive = idx.month.map({1: .2, 2: .1, 3: .2, 4: .3, 5: .2, 6: .2, 7: .3, 8: .5, 9: .4, 10: .9, 11: 1.0, 12: .8}).to_numpy()
    diwali_month = {2016: 10, 2017: 10, 2018: 11, 2019: 10, 2020: 11, 2021: 11, 2022: 10, 2023: 11, 2024: 10, 2025: 10}
    diwali = np.array([int(d.month == diwali_month[d.year]) for d in idx])
    lockdown = ((idx >= "2020-04-01") & (idx <= "2020-05-01")).astype(int)
    ad = np.round(10 + 0.08 * t + 8 * festive + rng.normal(0, 1, len(t)), 1)
    level = 200 + 2.0 * t + np.cumsum(rng.normal(0, 2.5, len(t)))
    sales = level * (1 + 0.30 * festive) * (1 + 0.15 * diwali) + 4 * ad + rng.normal(0, 10, len(t))
    sales = np.where(lockdown == 1, sales * 0.55, sales)
    df = pd.DataFrame({"sales": np.round(sales, 1), "ad_spend": ad, "festive": festive,
                       "diwali": diwali, "lockdown": lockdown}, index=idx)
    df.index.name = "month"
    df.loc[["2019-07-01", "2022-02-01", "2023-09-01"], "sales"] = np.nan
    return df

df = make_retail()
REGRESSORS = ["ad_spend", "diwali", "lockdown"]
data = df.reset_index().rename(columns={"month": "ds", "sales": "y"})[["ds", "y"] + REGRESSORS]
print(data.tail(3))

            ds      y  ad_spend  diwali  lockdown
117 2025-10-01  675.5      25.7       1         0
118 2025-11-01  628.4      28.4       0         0
119 2025-12-01  590.6      26.1       0         0


In [3]:
from prophet import Prophet
from prophet.serialize import model_to_json

def train_prophet(frame):
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                seasonality_mode="multiplicative", interval_width=0.90)
    for col in REGRESSORS:
        m.add_regressor(col)
    return m.fit(frame)

# 1) validate on 2025 before anything ships
holdout = data[data["ds"] >= "2025-01-01"]
model_v = train_prophet(data[data["ds"] < "2025-01-01"])
pred = model_v.predict(holdout[["ds"] + REGRESSORS])
mape_2025 = float(np.mean(np.abs(holdout["y"].values - pred["yhat"].values) / holdout["y"].values) * 100)

# 2) retrain on all history, then package model + metadata
model = train_prophet(data)
VERSION = "1.0.0"
os.makedirs(f"model/v{VERSION}", exist_ok=True)
with open(f"model/v{VERSION}/prophet_model.json", "w") as f:
    f.write(model_to_json(model))
meta = {"model": "prophet", "version": VERSION, "regressors": REGRESSORS,
        "trained_on": f"{data['ds'].min():%Y-%m} to {data['ds'].max():%Y-%m}",
        "holdout_mape_2025": round(mape_2025, 2), "interval": 0.90,
        "data_sha256": hashlib.sha256(pd.util.hash_pandas_object(data, index=False).values.tobytes()).hexdigest()[:16]}
json.dump(meta, open(f"model/v{VERSION}/metadata.json", "w"), indent=2)
print(json.dumps(meta, indent=2))

{
  "model": "prophet",
  "version": "1.0.0",
  "regressors": [
    "ad_spend",
    "diwali",
    "lockdown"
  ],
  "trained_on": "2016-01 to 2025-12",
  "holdout_mape_2025": 1.98,
  "interval": 0.9,
  "data_sha256": "97116e058bb9aac3"
}


## Part 2 · Serve it: a FastAPI forecast endpoint
The API loads the packaged model once at start-up, **validates every request** with Pydantic and returns the forecast **with its interval** and the model version.

In [4]:
%%writefile forecast_api.py
"""Retail sales forecast API: POST /forecast with the plan for each future month."""
import json, os
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from prophet.serialize import model_from_json

MODEL_DIR = os.getenv("MODEL_DIR", "model/v1.0.0")
with open(f"{MODEL_DIR}/prophet_model.json") as f:
    MODEL = model_from_json(f.read())
META = json.load(open(f"{MODEL_DIR}/metadata.json"))

app = FastAPI(title="Retail sales forecast API", version=META["version"])

class MonthPlan(BaseModel):
    month: str = Field(..., pattern=r"^\d{4}-\d{2}$", examples=["2026-01"])
    ad_spend: float = Field(..., ge=0, le=500, description="Planned ad spend, ₹ lakh")
    diwali: int = Field(0, ge=0, le=1)
    lockdown: int = Field(0, ge=0, le=1)

class ForecastRequest(BaseModel):
    months: list[MonthPlan] = Field(..., min_length=1, max_length=24)

@app.get("/health")
def health():
    return {"status": "ok", "model_version": META["version"]}

@app.post("/forecast")
def forecast(req: ForecastRequest):
    frame = pd.DataFrame([m.model_dump() for m in req.months])
    frame["ds"] = pd.to_datetime(frame.pop("month") + "-01")
    if frame["ds"].min() <= MODEL.history["ds"].max():
        raise HTTPException(status_code=422, detail="Forecast months must be after the training data.")
    out = MODEL.predict(frame[["ds"] + META["regressors"]])
    return {"model_version": META["version"], "interval": META["interval"],
            "forecast": [{"month": f"{r.ds:%Y-%m}", "sales": round(r.yhat, 1),
                          "lower": round(r.yhat_lower, 1), "upper": round(r.yhat_upper, 1)} for r in out.itertuples()]}

Overwriting forecast_api.py


In [5]:
from fastapi.testclient import TestClient
import importlib, forecast_api
importlib.reload(forecast_api)
client = TestClient(forecast_api.app)

plan = {"months": [{"month": "2026-10", "ad_spend": 24, "diwali": 0},
                   {"month": "2026-11", "ad_spend": 26, "diwali": 1}]}
print(client.get("/health").json())
r = client.post("/forecast", json=plan)
print(r.status_code, json.dumps(r.json(), indent=1))
print("bad request →", client.post("/forecast", json={"months": [{"month": "Oct-2026", "ad_spend": -5}]}).status_code)

{'status': 'ok', 'model_version': '1.0.0'}
200 {
 "model_version": "1.0.0",
 "interval": 0.9,
 "forecast": [
  {
   "month": "2026-10",
   "sales": 617.5,
   "lower": 602.2,
   "upper": 633.0
  },
  {
   "month": "2026-11",
   "sales": 705.8,
   "lower": 691.8,
   "upper": 720.7
  }
 ]
}
bad request → 422


In [6]:
%%writefile test_forecast_api.py
from fastapi.testclient import TestClient
from forecast_api import app

client = TestClient(app)

def test_health():
    assert client.get("/health").json()["status"] == "ok"

def test_forecast_has_interval():
    r = client.post("/forecast", json={"months": [{"month": "2026-03", "ad_spend": 15}]})
    row = r.json()["forecast"][0]
    assert r.status_code == 200 and row["lower"] <= row["sales"] <= row["upper"]

def test_rejects_bad_input():
    assert client.post("/forecast", json={"months": []}).status_code == 422
    assert client.post("/forecast", json={"months": [{"month": "2020-01", "ad_spend": 10}]}).status_code == 422

def test_quality_gate():
    import json
    assert json.load(open("model/v1.0.0/metadata.json"))["holdout_mape_2025"] < 10   # block a bad model

Overwriting test_forecast_api.py


In [7]:
# Run the tests exactly as a CI server would (sys.executable = this notebook's Python)
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "test_forecast_api.py"],
                        capture_output=True, text=True)
print(result.stdout[-400:] or result.stderr[-400:])
print("CI verdict:", "PASS — safe to build the image" if result.returncode == 0 else "FAIL — stop the release")

' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
    setattr(model, attribute, pd.Timedelta(seconds=model_dict[attribute]))

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
4 passed, 2 warnings in 1.38s

CI verdict: PASS — safe to build the image


## Part 3 · Ship it: the Dockerfile and the CI workflow
These files are written for you to use; this notebook does not run Docker. Build and run locally with
`docker build -t forecast-api .` then `docker run -p 8000:8000 forecast-api`, and open http://localhost:8000/docs.
A complete, tested reference repo with MLflow + Flask + CI is `mini-mlflow-flask-ci/` on the course GitHub.

In [8]:
%%writefile requirements-api.txt
prophet>=1.1
fastapi>=0.110
uvicorn>=0.29
pandas>=2.0

Overwriting requirements-api.txt


In [9]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt
COPY forecast_api.py .
COPY model/ model/
EXPOSE 8000
CMD ["uvicorn", "forecast_api:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [10]:
%%writefile forecast-ci.yml
# Save as .github/workflows/forecast-ci.yml in your repository
name: forecast-api-ci
on: [push, pull_request]
jobs:
  test-build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements-api.txt pytest httpx
      - run: pytest -q test_forecast_api.py          # tests + quality gate
      - run: docker build -t forecast-api:${{ github.sha }} .

Overwriting forecast-ci.yml


## Part 4 · Big data: forecasting IoT sensors at scale with PySpark
50 machines × 60 days of **hourly temperature** readings (72,000 rows). Spark spreads the work across cores here, or across a cluster in production, with the same code.

In [11]:
SPARK_OK = False
try:
    from pyspark.sql import SparkSession, functions as F, Window
    spark = (SparkSession.builder.master("local[*]").appName("apa-iot")
             .config("spark.sql.session.timeZone", "UTC")        # avoids time-zone lookup errors on some machines
             .config("spark.sql.shuffle.partitions", "8")
             .config("spark.driver.host", "127.0.0.1").getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    SPARK_OK = True
    print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism)
except Exception as e:
    print("Spark is not available here:", type(e).__name__, str(e)[:200])
    print("Install with: %pip install pyspark pyarrow  (Spark also needs Java 17+)")

Spark 4.3.0 | cores: 2


In [12]:
# Batch ingestion: generate the readings, land them as partitioned Parquet (the data-lake pattern), read back with Spark
rng = np.random.default_rng(21)
n_dev, hours = 50, 24 * 60
ts = pd.date_range("2025-01-01", periods=hours, freq="h")
iot = pd.DataFrame({"device_id": np.repeat(np.arange(n_dev), hours), "ts": np.tile(ts, n_dev)})
iot["temp_c"] = (60 + 0.1 * iot["device_id"] + 5 * np.sin(2 * np.pi * iot["ts"].dt.hour / 24)
                 + rng.normal(0, 1, len(iot))).round(2)
print(iot.shape)
if SPARK_OK:
    spark.createDataFrame(iot).write.mode("overwrite").partitionBy("device_id").parquet("lake/iot_hourly")
    sdf = spark.read.parquet("lake/iot_hourly")
    print("rows in the lake:", sdf.count(), "| partitions read:", sdf.rdd.getNumPartitions())

(72000, 3)


rows in the lake: 72000 | partitions read: 2


In [13]:
if SPARK_OK:
    w = Window.partitionBy("device_id").orderBy("ts")
    features = (sdf.withColumn("lag_1h", F.lag("temp_c", 1).over(w))
                   .withColumn("lag_24h", F.lag("temp_c", 24).over(w))
                   .withColumn("avg_24h", F.avg("temp_c").over(w.rowsBetween(-23, 0)))
                   .withColumn("hour", F.hour("ts"))
                   .dropna())
    features.select("device_id", "ts", "temp_c", "lag_1h", "lag_24h", "avg_24h").show(3)
    print("feature rows:", features.count())

+---------+-------------------+------+------+-------+-----------------+
|device_id|                 ts|temp_c|lag_1h|lag_24h|          avg_24h|
+---------+-------------------+------+------+-------+-----------------+
|        0|2025-01-02 00:00:00| 59.85| 58.71|  60.36|60.13708333333333|
|        0|2025-01-02 01:00:00| 62.19| 59.85|   62.8|60.11166666666667|
|        0|2025-01-02 02:00:00| 62.62| 62.19|  60.71|         60.19125|
+---------+-------------------+------+------+-------+-----------------+
only showing top 3 rows


feature rows: 70800


In [14]:
# One model per device, trained in parallel: each group arrives as a pandas DataFrame
def forecast_device(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("ts")
    t_end = pd.Timestamp(g["ts"].max())
    last_week = g[g["ts"] > t_end - pd.Timedelta(days=7)]
    profile = last_week.groupby(last_week["ts"].dt.hour)["temp_c"].mean()       # hourly profile of the last 7 days
    future = pd.date_range(t_end + pd.Timedelta(hours=1), periods=24, freq="h")
    return pd.DataFrame({"device_id": g["device_id"].iloc[0], "ts": future,
                         "forecast_c": profile.reindex(future.hour).to_numpy()})

if SPARK_OK:
    fc = sdf.groupBy("device_id").applyInPandas(forecast_device, schema="device_id int, ts timestamp, forecast_c double")
    fc_pd = fc.toPandas()
    print(fc_pd.shape, "→ 24-hour forecasts for", fc_pd["device_id"].nunique(), "devices")
    print(fc_pd.head(3))

(1200, 3) → 24-hour forecasts for 50 devices
   device_id                  ts  forecast_c
0          0 2025-03-02 00:00:00   60.122857
1          0 2025-03-02 01:00:00   61.392857
2          0 2025-03-02 02:00:00   62.531429


## Part 5 · Streaming ingestion with Structured Streaming
New sensor files land in a folder every few minutes. Spark treats the folder as an **unbounded table**: the same aggregation code works for batch and stream.
`trigger(availableNow=True)` processes what has arrived and stops, which is handy for a notebook and for scheduled micro-batches.

In [15]:
import shutil
if SPARK_OK:
    shutil.rmtree("stream_in", ignore_errors=True); shutil.rmtree("stream_ckpt", ignore_errors=True)
    os.makedirs("stream_in")
    for k in range(3):                                   # three micro-batches arrive
        part = iot[iot["ts"] >= ts[-3]].sample(150, random_state=k)
        part.assign(ts=part["ts"].dt.strftime("%Y-%m-%d %H:%M:%S")).to_csv(f"stream_in/batch_{k}.csv", index=False)
    schema = "device_id INT, ts TIMESTAMP, temp_c DOUBLE"
    stream = spark.readStream.schema(schema).option("header", True).csv("stream_in")
    live = stream.groupBy("device_id").agg(F.count("*").alias("readings"), F.round(F.avg("temp_c"), 2).alias("avg_temp"))
    q = (live.writeStream.format("memory").queryName("live_device_stats").outputMode("complete")
             .option("checkpointLocation", "stream_ckpt").trigger(availableNow=True).start())
    q.awaitTermination()
    spark.sql("SELECT * FROM live_device_stats ORDER BY readings DESC").show(5)

+---------+--------+--------+
|device_id|readings|avg_temp|
+---------+--------+--------+
|       14|       9|   59.42|
|       38|       9|   62.56|
|       13|       9|   58.53|
|       12|       9|    59.0|
|       46|       9|   62.64|
+---------+--------+--------+
only showing top 5 rows


## Part 6 · Monitoring: is the world still the one we trained on?
**Data drift**: the inputs change. **Concept drift**: the relationship changes. Two common checks on a feature:
* **PSI** (population stability index): a common rule of thumb reads < 0.10 as stable, 0.10–0.25 as watch, > 0.25 as act.
* **KS test**: are two samples from the same distribution? A small p-value says no.

Scenario: after a firmware update in week 9, devices 0–9 read about 3 °C hotter.

In [16]:
from scipy.stats import ks_2samp

def psi(reference, current, bins=10):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    ref = np.histogram(reference, edges)[0] / len(reference)
    cur = np.histogram(current, edges)[0] / len(current)
    ref, cur = np.clip(ref, 1e-6, None), np.clip(cur, 1e-6, None)
    return float(np.sum((cur - ref) * np.log(cur / ref)))

reference = iot[iot["ts"] < "2025-02-01"]
current = iot[iot["ts"] >= "2025-02-23"].copy()
current.loc[current["device_id"] < 10, "temp_c"] += 3.0          # the firmware shift
rows = []
for dev in [0, 5, 20, 40]:
    r, c = reference.loc[reference.device_id == dev, "temp_c"], current.loc[current.device_id == dev, "temp_c"]
    rows.append({"device": dev, "PSI": round(psi(r, c), 3), "KS p-value": ks_2samp(r, c).pvalue})
drift = pd.DataFrame(rows)
drift["verdict"] = pd.cut(drift["PSI"], [-1, 0.10, 0.25, 99], labels=["stable", "watch", "act"])
drift

,device,PSI,KS p-value,verdict
0,0,2.837,2.414605e-11,act
1,5,2.785,5.061859e-12,act
2,20,0.092,8.962498e-01,stable
3,40,0.033,9.680277e-01,stable


In [17]:
# The same check with Evidently's data-drift preset (the course's monitoring tool)
try:
    from evidently import Report
    from evidently.presets import DataDriftPreset
    ref_tbl = reference.pivot_table(index="ts", columns="device_id", values="temp_c").iloc[:, :12]
    cur_tbl = current.pivot_table(index="ts", columns="device_id", values="temp_c").iloc[:, :12]
    ref_tbl.columns = cur_tbl.columns = [f"dev_{c}" for c in ref_tbl.columns]
    snapshot = Report([DataDriftPreset()]).run(current_data=cur_tbl, reference_data=ref_tbl)
    snapshot.save_html("drift_report.html")
    print(snapshot.dict()["metrics"][0]["value"], "→ open drift_report.html for the full report")
except Exception as e:
    print("Evidently unavailable here:", type(e).__name__, str(e)[:160])
    print("Try: %pip install evidently  (it needs pandas 2.x). The PSI and KS checks above do the same job.")

{'count': 10.0, 'share': 0.8333333333333334} → open drift_report.html for the full report


## Part 7 · A retraining pipeline with a champion–challenger gate
Retraining is not "fit again and hope". The pipeline **detects** (drift or error), **retrains**, **validates the challenger against the champion**
on the same recent data, and **promotes only if it wins**. Prefect turns plain Python functions into a tracked, schedulable flow
(Airflow does the same job with DAGs).

In [18]:
def detect(psi_score, recent_mape, psi_limit=0.25, mape_limit=8.0):
    return psi_score > psi_limit or recent_mape > mape_limit

def champion_challenger(train_frame, valid_frame, champion):
    challenger = train_prophet(train_frame)
    score = lambda m: float(np.mean(np.abs(valid_frame["y"].values - m.predict(valid_frame[["ds"] + REGRESSORS])["yhat"].values)
                                    / valid_frame["y"].values) * 100)
    return challenger, score(champion), score(challenger)

os.environ.setdefault("PREFECT_LOGGING_LEVEL", "WARNING")      # keep the notebook output readable
os.environ.setdefault("PREFECT_SERVER_ANALYTICS_ENABLED", "false")
try:
    from prefect import flow, task
    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
    detect_t, cc_t = task(detect), task(champion_challenger)

    @flow(name="retail-retraining")
    def retraining_flow(psi_score: float, recent_mape: float):
        if not detect_t(psi_score, recent_mape):
            return "no action: keep the champion"
        valid = data[data["ds"] >= "2025-07-01"]
        challenger, champ_mape, chall_mape = cc_t(data[data["ds"] < "2025-07-01"], valid, model_v)
        verdict = "promote challenger" if chall_mape < champ_mape else "keep champion"
        return f"champion MAPE {champ_mape:.1f}% vs challenger {chall_mape:.1f}% → {verdict}"

    print(retraining_flow(psi_score=0.05, recent_mape=3.0))
    print(retraining_flow(psi_score=0.40, recent_mape=3.0))
except ImportError:
    print("Prefect not installed (%pip install prefect). The same logic as plain Python:")
    print("detect(psi=0.40, mape=3.0) →", detect(0.40, 3.0))

no action: keep the champion


champion MAPE 0.9% vs challenger 1.5% → keep champion


In [19]:
if SPARK_OK:
    spark.stop()
print("Artefacts created:", sorted(p for p in os.listdir(".") if p.endswith((".py", ".txt", ".yml", ".html")) or p in ("Dockerfile", "model")))

Artefacts created: ['Dockerfile', 'drift_report.html', 'forecast-ci.yml', 'forecast_api.py', 'model', 'requirements-api.txt', 'test_forecast_api.py']


### What to take away
* A model is deployable only with its **metadata, tests and a quality gate**.
* The API **validates inputs**, returns **intervals** and its **version**.
* Spark lets the **same code** handle batch, stream and thousands of per-entity models.
* Monitor **inputs (PSI, KS)** and **outcomes (error)**; retrain through a **champion–challenger** gate, never blindly.